**NOMS et Prénoms :** 

# TP 2 — Systèmes linéaires et interpolation

**Analyse numérique — ESTP PGE1 S6 — 2025-2026**

---

## Consignes

- Ce notebook est **le sujet et le rendu**. Complétez les cellules de code marquées `# À COMPLÉTER` et répondez aux questions dans les cellules prévues.
- Chaque cellule peut être exécutée séparément en cliquant sur **Maj+Entrée**.
Lorsque vous lancez l'exécution d'une cellule, observez bien le symbole entre les crochets en haut à gauche. Si c'est une étoile, l'exécution est en cours !
- Le notebook doit **s'exécuter sans erreur** de bout en bout.
- Les réponses aux questions doivent être **argumentées** (2-3 phrases minimum).

## Barème

| Partie | Contenu | Points |
|--------|---------|--------|
| 1 | Interpolation de Lagrange | /4 |
| 2 | Décomposition LU et résolution directe | /4 |
| 3 | Méthodes itératives : Jacobi et Gauss-Seidel | /5 |
| 4 | Comparaison directes vs itératives et temps de calcul | /4 |
| 5 | Perturbations et conditionnement | /3 |
| **Total** | | **/20** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 12})

---

# Partie 1 — Interpolation de Lagrange (/4 pts)

En TD1, on a interpolé des données par des polynômes de Lagrange. Le polynôme interpolateur de degré $n$ passant par les $n+1$ points $(x_0, y_0), \ldots, (x_n, y_n)$ s'écrit :

$$P(x) = \sum_{j=0}^{n} y_j \, L_j(x), \qquad L_j(x) = \prod_{\substack{k=0 \\ k \neq j}}^{n} \frac{x - x_k}{x_j - x_k}$$

Chaque $L_j$ est un **polynôme de base de Lagrange** : il vaut 1 au nœud $x_j$ et 0 aux autres nœuds.

## Exercice 1.1 — Implémenter l'interpolation de Lagrange (1,5 pt)

Compléter la fonction ci-dessous. Elle prend en entrée :
- `x_nodes` : tableau des abscisses des nœuds $(x_0, \ldots, x_n)$
- `y_nodes` : tableau des ordonnées $(y_0, \ldots, y_n)$
- `x_eval` : tableau des points où évaluer le polynôme

Elle renvoie un tableau NumPy contenant les valeurs du polynôme interpolateur aux points `x_eval`.

**Indication :** pour chaque point `x` de `x_eval`, calculer chaque $L_j(x)$ à l'aide d'un produit, puis sommer $y_j \cdot L_j(x)$.

In [ ]:
def lagrange_interp(x_nodes, y_nodes, x_eval):
    """Interpolation de Lagrange."""
    n = len(x_nodes)
    result = np.zeros_like(x_eval, dtype=float)

    for j in range(n):
        # Calcul du polynôme de base L_j
        Lj = np.ones_like(x_eval, dtype=float)
        for k in range(n):
            if k != j:
                Lj *= ...  # À COMPLÉTER : (x_eval - x_k) / (x_j - x_k)
        result += ...  # À COMPLÉTER : y_j * L_j

    return result

Pour vérifier votre implémentation, lancez la cellule suivante ! On interpole les points $(0,1)$, $(1,3)$, $(2,7)$ qui correspondent au polynôme $P(x) = x^2 + x + 1$.

In [ ]:
# --- Vérification automatique ---
x_test = np.array([0.0, 1.0, 2.0])
y_test = np.array([1.0, 3.0, 7.0])
x_ev = np.array([0.5, 1.5, 0.0, 2.0])
attendu = x_ev**2 + x_ev + 1

resultat = lagrange_interp(x_test, y_test, x_ev)
erreur = np.max(np.abs(resultat - attendu))

print(f"Valeurs calculées : {resultat}")
print(f"Valeurs attendues : {attendu}")
print(f"Erreur max : {erreur:.2e}")
assert erreur < 1e-10, "L'erreur semble trop grande, vérifiez votre formule."
print("✓ Test réussi !")

## Exercice 1.2 — Application : profil d'une façade (1 pt)

Un géomètre a relevé la hauteur d'un arc de façade en 7 points :

| Position $x$ (m) | 0 | 2 | 4 | 6 | 8 | 10 | 12 |
|---|---|---|---|---|---|---|---|
| Hauteur $y$ (m) | 0 | 3,5 | 6,0 | 7,2 | 6,8 | 4,0 | 0 |

**À faire :**
1. Appeler `lagrange_interp` pour interpoler le profil sur 200 points régulièrement espacés entre 0 et 12.
2. Estimer la **surface** de la façade en utilisant `np.trapz` sur le profil interpolé.

In [ ]:
x_facade = np.array([0, 2, 4, 6, 8, 10, 12], dtype=float)
y_facade = np.array([0, 3.5, 6.0, 7.2, 6.8, 4.0, 0], dtype=float)

x_fin = np.linspace(0, 12, 200)
y_interp = ...  # À COMPLÉTER : appeler lagrange_interp

surface = ...  # À COMPLÉTER : utiliser np.trapz(y_interp, x_fin)
surface_brute = np.trapz(y_facade, x_facade)

print(f"Surface estimée (interpolation fine) : {surface:.2f} m²")
print(f"Surface estimée (trapèzes sur 7 pts) : {surface_brute:.2f} m²")

In [ ]:
# --- Tracé fourni ---
plt.figure()
plt.fill_between(x_fin, 0, y_interp, alpha=0.15, color="C0")
plt.plot(x_fin, y_interp, "C0", linewidth=2, label="Interpolation de Lagrange")
plt.plot(x_facade, y_facade, "ko", markersize=8, label="Mesures")
plt.xlabel("Position (m)")
plt.ylabel("Hauteur (m)")
plt.title(f"Profil de façade — surface ≈ {surface:.1f} m²")
plt.legend()
plt.grid(True)
plt.ylim(bottom=-0.5)
plt.show()

## Exercice 1.3 — Le phénomène de Runge (1 pt)

On interpole la fonction $f(x) = \dfrac{1}{1 + 25x^2}$ sur $[-1, 1]$ avec des nœuds **équidistants**. Lorsque le nombre de nœuds augmente, le polynôme interpolateur ne converge pas partout vers $f$ : c'est le **phénomène de Runge**.

Le code ci-dessous trace l'interpolation pour $n = 5$, $10$ et $15$ nœuds. **Exécuter** et répondre aux questions.

In [ ]:
def runge(x):
    return 1 / (1 + 25 * x**2)

x_plot = np.linspace(-1, 1, 500)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for idx, n in enumerate([5, 10, 15]):
    x_nodes = np.linspace(-1, 1, n + 1)
    y_nodes = runge(x_nodes)
    y_lag = lagrange_interp(x_nodes, y_nodes, x_plot)

    axes[idx].plot(x_plot, runge(x_plot), "k-", linewidth=2, label="f(x)")
    axes[idx].plot(x_plot, y_lag, "C0", linewidth=1.5, label=f"Lagrange (n={n})")
    axes[idx].plot(x_nodes, y_nodes, "ko", markersize=5)
    axes[idx].set_ylim(-1.5, 2)
    axes[idx].set_xlabel("x")
    axes[idx].set_title(f"n = {n} nœuds")
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True)

plt.suptitle("Phénomène de Runge — nœuds équidistants", fontsize=14)
plt.tight_layout()
plt.show()

## Question 1.4 — Interprétation (0,5 pt)

1. Que se passe-t-il aux bords de l'intervalle quand $n$ augmente ?
2. En quoi cela incite-t-il à la prudence quand on interpole des mesures expérimentales avec beaucoup de points ?

**Votre réponse :** *(double-cliquez pour éditer)*


---

# Partie 2 — Décomposition LU et résolution directe (/4 pts)

En TD2, on a résolu des systèmes linéaires $A x = b$ par le **pivot de Gauss**, puis on a vu que cette élimination revenait à factoriser $A = L U$ où :
- $L$ est triangulaire **inférieure** avec des 1 sur la diagonale,
- $U$ est triangulaire **supérieure**.

La résolution se fait alors en deux étapes :
1. **Descente** : résoudre $L y = b$ (substitution avant)
2. **Remontée** : résoudre $U x = y$ (substitution arrière)

## Exercice 2.1 — Implémenter la décomposition LU (2 pts)

Compléter la fonction ci-dessous. L'algorithme de Doolittle (sans pivotement) procède colonne par colonne :

Pour $k = 0, 1, \ldots, n-1$ :
- Éléments de $U$ (ligne $k$) : $U_{k,j} = A_{k,j} - \displaystyle\sum_{s=0}^{k-1} L_{k,s}\, U_{s,j}$ pour $j \geq k$
- Éléments de $L$ (colonne $k$) : $L_{i,k} = \dfrac{1}{U_{k,k}}\left(A_{i,k} - \displaystyle\sum_{s=0}^{k-1} L_{i,s}\, U_{s,k}\right)$ pour $i > k$

In [ ]:
def decomposition_lu(A):
    """Décomposition LU (Doolittle, sans pivotement)."""
    n = A.shape[0]
    L = np.eye(n)
    U = np.zeros((n, n))

    for k in range(n):
        # Ligne k de U
        for j in range(k, n):
            U[k, j] = ...  # À COMPLÉTER
        # Colonne k de L
        for i in range(k + 1, n):
            L[i, k] = ...  # À COMPLÉTER

    return L, U

Pour vérifier, on factorise une matrice $3 \times 3$ et on vérifie que $L \cdot U = A$.

In [ ]:
# --- Vérification automatique ---
A_test = np.array([[2, 1, 1],
                   [4, 3, 3],
                   [8, 7, 9]], dtype=float)

L_test, U_test = decomposition_lu(A_test)

print("L =")
print(L_test)
print("\nU =")
print(U_test)
print(f"\nL @ U =\n{L_test @ U_test}")
print(f"\nA     =\n{A_test}")

erreur_lu = np.max(np.abs(L_test @ U_test - A_test))
print(f"\nErreur max |L·U - A| : {erreur_lu:.2e}")
assert erreur_lu < 1e-10, "La décomposition LU semble incorrecte."
assert np.allclose(np.diag(L_test), 1), "La diagonale de L doit valoir 1."
print("✓ Test réussi !")

## Exercice 2.2 — Résolution par descente-remontée (1 pt)

Compléter la fonction qui résout $Ax = b$ en utilisant la décomposition $A = LU$ :
1. **Descente** : $y_i = b_i - \sum_{j=0}^{i-1} L_{i,j}\, y_j$
2. **Remontée** : $x_i = \dfrac{1}{U_{i,i}} \left(y_i - \sum_{j=i+1}^{n-1} U_{i,j}\, x_j\right)$

In [ ]:
def resoudre_lu(L, U, b):
    """Résolution de LUx = b par descente puis remontée."""
    n = len(b)
    y = np.zeros(n)
    x = np.zeros(n)

    # Descente : Ly = b
    for i in range(n):
        y[i] = ...  # À COMPLÉTER

    # Remontée : Ux = y
    for i in range(n - 1, -1, -1):
        x[i] = ...  # À COMPLÉTER

    return x

In [ ]:
# --- Vérification automatique ---
b_test = np.array([1.0, 5.0, 3.0])
x_lu = resoudre_lu(L_test, U_test, b_test)
x_ref = np.linalg.solve(A_test, b_test)

print(f"Solution LU      : {x_lu}")
print(f"Solution NumPy   : {x_ref}")

erreur_sol = np.max(np.abs(x_lu - x_ref))
print(f"Erreur max : {erreur_sol:.2e}")
assert erreur_sol < 1e-10, "La résolution semble incorrecte."
print("✓ Test réussi !")

## Exercice 2.3 — Application : déformation d'une poutre (/1 pt)

On modélise la flèche $u(x)$ d'une poutre simplement appuyée de longueur $L = 10$ m sous une charge uniforme $q$ par l'équation :

$$-u''(x) = \frac{q}{EI}, \qquad u(0) = 0, \quad u(L) = 0$$

En discrétisant avec $n$ points intérieurs et un pas $h = L / (n+1)$, on obtient le système linéaire $K u = f$ avec la **matrice de rigidité** tridiagonale :

$$K = \frac{1}{h^2} \begin{pmatrix} 2 & -1 & & \\ -1 & 2 & -1 & \\ & \ddots & \ddots & \ddots \\ & & -1 & 2 \end{pmatrix}, \qquad f_i = \frac{q}{EI}$$

La solution exacte est connue : $u(x) = \dfrac{q}{2\,EI}\,x\,(L - x)$.

**À faire :** compléter le code pour construire la matrice $K$ et le vecteur $f$, puis résoudre avec votre décomposition LU.

In [ ]:
L_poutre = 10.0
n_poutre = 20
q_sur_EI = 1.0
h = L_poutre / (n_poutre + 1)

# Construction de la matrice de rigidité K (n x n)
K = np.zeros((n_poutre, n_poutre))
for i in range(n_poutre):
    K[i, i] = ...      # À COMPLÉTER : coefficient diagonal
    if i > 0:
        K[i, i-1] = ... # À COMPLÉTER : sous-diagonale
    if i < n_poutre - 1:
        K[i, i+1] = ... # À COMPLÉTER : sur-diagonale

# Second membre
f_poutre = ...  # À COMPLÉTER : vecteur de taille n_poutre rempli de q_sur_EI

# Résolution par LU
L_K, U_K = decomposition_lu(K)
u_num = resoudre_lu(L_K, U_K, f_poutre)

# Reconstitution avec les conditions aux limites u(0) = u(L) = 0
x_poutre = np.linspace(0, L_poutre, n_poutre + 2)
u_complet = np.zeros(n_poutre + 2)
u_complet[1:-1] = u_num

In [ ]:
# --- Tracé fourni ---
x_exact = np.linspace(0, L_poutre, 300)
u_exact = (q_sur_EI / 2) * x_exact * (L_poutre - x_exact)

plt.figure()
plt.plot(x_exact, u_exact, "k-", linewidth=2, label="Solution exacte")
plt.plot(x_poutre, u_complet, "o--", color="C0", markersize=5, label=f"LU ({n_poutre} pts)")
plt.xlabel("Position x (m)")
plt.ylabel("Flèche u (m)")
plt.title("Déformation d'une poutre — résolution par LU")
plt.legend()
plt.grid(True)
plt.show()

erreur_poutre = np.max(np.abs(u_complet - (q_sur_EI / 2) * x_poutre * (L_poutre - x_poutre)))
print(f"Erreur max : {erreur_poutre:.4e} m")

---

# Partie 3 — Méthodes itératives : Jacobi et Gauss-Seidel (/5 pts)

En TD3, on a étudié les méthodes itératives pour résoudre $Ax = b$. Contrairement aux méthodes directes (Gauss, LU), on part d'un vecteur initial $x^{(0)}$ et on construit une suite $x^{(0)}, x^{(1)}, \ldots$ qui converge vers la solution.

**Méthode de Jacobi :** on isole $x_i$ dans la $i$-ème équation en utilisant les valeurs de l'itération précédente :

$$x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{\substack{j=0 \\ j \neq i}}^{n-1} a_{ij}\, x_j^{(k)} \right)$$

**Méthode de Gauss-Seidel :** même idée, mais on utilise les valeurs **déjà mises à jour** dès qu'elles sont disponibles :

$$x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j=0}^{i-1} a_{ij}\, x_j^{(k+1)} - \sum_{j=i+1}^{n-1} a_{ij}\, x_j^{(k)} \right)$$

## Exercice 3.1 — Méthode de Jacobi (2 pts)

Compléter la fonction ci-dessous. Elle renvoie :
- `x` : la solution approchée
- `residus` : la liste des normes du résidu $\|b - Ax^{(k)}\|$ à chaque itération (pour tracer la convergence)

In [ ]:
def jacobi(A, b, x0, tol=1e-8, max_iter=1000):
    """Méthode de Jacobi pour Ax = b."""
    n = len(b)
    x = x0.copy()
    residus = []

    for k in range(max_iter):
        x_new = np.zeros(n)
        for i in range(n):
            s = 0.0
            for j in range(n):
                if j != i:
                    s += ...  # À COMPLÉTER : a_ij * x_j^(k)
            x_new[i] = ...   # À COMPLÉTER : (b_i - s) / a_ii

        residu = np.linalg.norm(b - A @ x_new)
        residus.append(residu)

        if residu < tol:
            return x_new, residus

        x = x_new

    print(f"⚠ Jacobi : pas de convergence après {max_iter} itérations (résidu = {residu:.2e})")
    return x, residus

In [ ]:
# --- Vérification automatique ---
A_iter = np.array([[4, 1, 0],
                   [1, 3, 1],
                   [0, 1, 4]], dtype=float)
b_iter = np.array([5.0, 5.0, 5.0])
x0_iter = np.zeros(3)

x_jac, res_jac = jacobi(A_iter, b_iter, x0_iter)
x_exact_iter = np.linalg.solve(A_iter, b_iter)

print(f"Solution Jacobi : {x_jac}")
print(f"Solution exacte : {x_exact_iter}")
print(f"Erreur max : {np.max(np.abs(x_jac - x_exact_iter)):.2e}")
print(f"Convergence en {len(res_jac)} itérations")
assert np.max(np.abs(x_jac - x_exact_iter)) < 1e-6, "La solution Jacobi semble incorrecte."
print("✓ Test réussi !")

## Exercice 3.2 — Méthode de Gauss-Seidel (1,5 pt)

Compléter la fonction ci-dessous. La seule différence avec Jacobi est que l'on utilise les valeurs **déjà calculées** $x_j^{(k+1)}$ pour $j < i$.

**Question de réflexion :** pourquoi n'a-t-on pas besoin d'un vecteur `x_new` séparé dans Gauss-Seidel, alors qu'il est nécessaire dans Jacobi ?

In [ ]:
def gauss_seidel(A, b, x0, tol=1e-8, max_iter=1000):
    """Méthode de Gauss-Seidel pour Ax = b."""
    n = len(b)
    x = x0.copy()
    residus = []

    for k in range(max_iter):
        for i in range(n):
            s = 0.0
            for j in range(n):
                if j != i:
                    s += ...  # À COMPLÉTER : a_ij * x_j (déjà mis à jour si j < i)
            x[i] = ...       # À COMPLÉTER : (b_i - s) / a_ii

        residu = np.linalg.norm(b - A @ x)
        residus.append(residu)

        if residu < tol:
            return x.copy(), residus

    print(f"⚠ Gauss-Seidel : pas de convergence après {max_iter} itérations (résidu = {residu:.2e})")
    return x.copy(), residus

In [ ]:
# --- Vérification automatique ---
x_gs, res_gs = gauss_seidel(A_iter, b_iter, np.zeros(3))

print(f"Solution Gauss-Seidel : {x_gs}")
print(f"Solution exacte       : {x_exact_iter}")
print(f"Erreur max : {np.max(np.abs(x_gs - x_exact_iter)):.2e}")
print(f"Convergence en {len(res_gs)} itérations")
assert np.max(np.abs(x_gs - x_exact_iter)) < 1e-6, "La solution Gauss-Seidel semble incorrecte."
print("✓ Test réussi !")

## Exercice 3.3 — Convergence comparée (1,5 pt)

On applique Jacobi et Gauss-Seidel au **système de la poutre** (matrice de rigidité $K$, second membre $f$). Le code ci-dessous trace l'évolution du résidu en fonction du nombre d'itérations.

**À faire :**
1. Compléter les appels à `jacobi` et `gauss_seidel` avec la matrice $K$ et le vecteur $f$ de la partie 2.
2. Exécuter et répondre : lequel converge le plus vite ? De combien d'itérations a-t-on besoin pour chaque méthode ?

In [ ]:
x0_poutre = np.zeros(n_poutre)

x_jac_poutre, res_jac_poutre = ...  # À COMPLÉTER : jacobi(K, f_poutre, ...)
x_gs_poutre, res_gs_poutre = ...    # À COMPLÉTER : gauss_seidel(K, f_poutre, ...)

print(f"Jacobi       : convergence en {len(res_jac_poutre)} itérations")
print(f"Gauss-Seidel : convergence en {len(res_gs_poutre)} itérations")

# --- Tracé fourni ---
plt.figure()
plt.semilogy(res_jac_poutre, "C0", linewidth=2, label=f"Jacobi ({len(res_jac_poutre)} it.)")
plt.semilogy(res_gs_poutre, "C1", linewidth=2, label=f"Gauss-Seidel ({len(res_gs_poutre)} it.)")
plt.xlabel("Itération")
plt.ylabel("Résidu ‖b − Ax‖")
plt.title("Convergence des méthodes itératives — poutre")
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

## Question 3.4 — Dominance diagonale et convergence (1 pt bonus)

On rappelle qu'une matrice est à **diagonale strictement dominante** si, pour chaque ligne :

$$|a_{ii}| > \sum_{\substack{j=0 \\ j \neq i}}^{n-1} |a_{ij}|$$

1. La matrice $K$ de la poutre est-elle à diagonale strictement dominante ? Vérifiez-le sur un exemple.
2. Qu'est-ce que cela garantit pour la convergence de Jacobi et Gauss-Seidel ?

**Votre réponse :** *(double-cliquez pour éditer)*


In [ ]:
# Vérification de la dominance diagonale de K
for i in range(min(5, n_poutre)):
    diag = abs(K[i, i])
    hors_diag = sum(abs(K[i, j]) for j in range(n_poutre) if j != i)
    status = "✓" if diag > hors_diag else "✗"
    print(f"Ligne {i:2d} : |a_ii| = {diag:.4f},  Σ|a_ij| = {hors_diag:.4f}  {status}")

print("...")

---

# Partie 4 — Comparaison directes vs itératives (/4 pts)

En pratique, le choix entre méthode directe et méthode itérative dépend de la **taille** du système, de sa **structure** (creuse ou dense) et de la **précision** souhaitée. On va maintenant mesurer et comparer les temps de calcul.

## Exercice 4.1 — Chronomètre sur la poutre (1,5 pt)

On résout le système de la poutre ($n = 100$ points intérieurs) par quatre méthodes :
- votre décomposition LU,
- `np.linalg.solve` (la routine optimisée de NumPy),
- Jacobi,
- Gauss-Seidel.

**À faire :** compléter les appels puis observer les temps. Lequel est le plus rapide ?

In [ ]:
def construire_systeme_poutre(n):
    """Construit K et f pour la poutre de taille n."""
    h = L_poutre / (n + 1)
    K = np.zeros((n, n))
    for i in range(n):
        K[i, i] = 2 / h**2
        if i > 0:
            K[i, i-1] = -1 / h**2
        if i < n - 1:
            K[i, i+1] = -1 / h**2
    f = np.full(n, q_sur_EI)
    return K, f


n_chrono = 100
K100, f100 = construire_systeme_poutre(n_chrono)
x0_100 = np.zeros(n_chrono)

# LU maison
t0 = time.time()
L100, U100 = decomposition_lu(K100)
x_lu100 = resoudre_lu(L100, U100, f100)
t_lu = time.time() - t0

# NumPy
t0 = time.time()
x_np100 = np.linalg.solve(K100, f100)
t_np = time.time() - t0

# Jacobi
t0 = time.time()
x_jac100, _ = ...  # À COMPLÉTER : jacobi(K100, f100, x0_100)
t_jac = time.time() - t0

# Gauss-Seidel
t0 = time.time()
x_gs100, _ = ...   # À COMPLÉTER : gauss_seidel(K100, f100, x0_100)
t_gs = time.time() - t0

print(f"{'Méthode':<20} {'Temps (ms)':>12} {'Erreur max':>12}")
print("-" * 46)
for nom, t, x_sol in [("LU maison", t_lu, x_lu100),
                       ("np.linalg.solve", t_np, x_np100),
                       ("Jacobi", t_jac, x_jac100),
                       ("Gauss-Seidel", t_gs, x_gs100)]:
    err = np.max(np.abs(x_sol - x_np100))
    print(f"{nom:<20} {t*1000:>10.2f} ms {err:>12.2e}")

**Votre réponse :** quelle méthode est la plus rapide ? Pourquoi `np.linalg.solve` est-il si rapide par rapport à votre LU ?

*(double-cliquez pour éditer)*


## Exercice 4.2 — Passage à l'échelle (1,5 pt)

On fait varier la taille $n$ du système et on mesure le temps de chaque méthode. Le code ci-dessous effectue les mesures.

**À faire :**
1. Compléter les lignes marquées pour chronométrer Jacobi et Gauss-Seidel.
2. Exécuter une première fois avec `tol=1e-6` et `max_iter=1000`.
3. Répondre : que remarquez-vous sur les temps de Jacobi/Gauss-Seidel et sur leur convergence éventuelle ?
4. Refaire ensuite avec une tolérance plus grande (`tol=1e-4`) et `max_iter=100000`.
5. Répondre à nouveau : que changent ces paramètres sur les temps mesurés et la comparaison des méthodes ?

In [ ]:
tailles = [10, 20, 30, 40, 50, 100, 200]

# 1) Première mesure : tolérance "stricte" et max_iter modéré
tol_1 = 1e-6
max_iter_1 = 1000

temps_lu_1 = []
temps_np_1 = []
temps_jac_1 = []
temps_gs_1 = []

for n in tailles:
    Kn, fn = construire_systeme_poutre(n)
    x0n = np.zeros(n)

    t0 = time.time()
    Ln, Un = decomposition_lu(Kn)
    _ = resoudre_lu(Ln, Un, fn)
    temps_lu_1.append(time.time() - t0)

    t0 = time.time()
    _ = np.linalg.solve(Kn, fn)
    temps_np_1.append(time.time() - t0)

    t0 = time.time()
    _, _ = jacobi(Kn, fn, x0n, tol=tol_1, max_iter=max_iter_1)
    temps_jac_1.append(time.time() - t0)

    t0 = time.time()
    _, _ = gauss_seidel(Kn, fn, x0n, tol=tol_1, max_iter=max_iter_1)
    temps_gs_1.append(time.time() - t0)

    print(f"[Config 1] n = {n:>4d} : LU = {temps_lu_1[-1]*1000:>8.1f} ms, "
          f"NumPy = {temps_np_1[-1]*1000:>6.2f} ms, "
          f"Jacobi = {temps_jac_1[-1]*1000:>8.1f} ms, "
          f"GS = {temps_gs_1[-1]*1000:>8.1f} ms")

print("\nQuestion intermédiaire : que constatez-vous sur la convergence et la comparaison des temps ?")

# 2) Seconde mesure : tolérance plus grande et max_iter augmenté
tol_2 = 1e-4
max_iter_2 = 100000

temps_lu_m = []
temps_np_m = []
temps_jac_m = []
temps_gs_m = []

for n in tailles:
    Kn, fn = construire_systeme_poutre(n)
    x0n = np.zeros(n)

    t0 = time.time()
    Ln, Un = decomposition_lu(Kn)
    _ = resoudre_lu(Ln, Un, fn)
    temps_lu_m.append(time.time() - t0)

    t0 = time.time()
    _ = np.linalg.solve(Kn, fn)
    temps_np_m.append(time.time() - t0)

    t0 = time.time()
    _, _ = jacobi(Kn, fn, x0n, tol=tol_2, max_iter=max_iter_2)
    temps_jac_m.append(time.time() - t0)

    t0 = time.time()
    _, _ = gauss_seidel(Kn, fn, x0n, tol=tol_2, max_iter=max_iter_2)
    temps_gs_m.append(time.time() - t0)

    print(f"[Config 2] n = {n:>4d} : LU = {temps_lu_m[-1]*1000:>8.1f} ms, "
          f"NumPy = {temps_np_m[-1]*1000:>6.2f} ms, "
          f"Jacobi = {temps_jac_m[-1]*1000:>8.1f} ms, "
          f"GS = {temps_gs_m[-1]*1000:>8.1f} ms")

In [ ]:
# --- Tracé fourni ---
plt.figure(figsize=(9, 5))
plt.loglog(tailles, temps_lu_m, "o-", color="C0", linewidth=2, label="LU maison")
plt.loglog(tailles, temps_np_m, "s-", color="C2", linewidth=2, label="np.linalg.solve")
plt.loglog(tailles, temps_jac_m, "^-", color="C1", linewidth=2, label="Jacobi")
plt.loglog(tailles, temps_gs_m, "v-", color="C3", linewidth=2, label="Gauss-Seidel")
plt.xlabel("Taille du système n")
plt.ylabel("Temps de calcul (s)")
plt.title("Passage à l'échelle — directes vs itératives")
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

# Mesure des pentes
for nom, temps in [("LU maison", temps_lu_m), ("np.linalg.solve", temps_np_m),
                    ("Jacobi", temps_jac_m), ("Gauss-Seidel", temps_gs_m)]:
    pente = np.polyfit(np.log(tailles), np.log(temps), 1)[0]
    print(f"{nom:<20} : pente ≈ {pente:.2f}")

## Question 4.3 — Interprétation (1 pt)

1. Première configuration (`tol=1e-6`, `max_iter=1000`) : que constatez-vous sur les temps de Jacobi/Gauss-Seidel et sur leur convergence ?
2. Seconde configuration (`tol=1e-4`, `max_iter=100000`) : qu'est-ce qui change dans les temps et dans la comparaison entre méthodes ?
3. Quelle est la complexité théorique de la décomposition LU sur une matrice dense $n \times n$ ? Est-ce cohérent avec la pente observée ?
4. Pour une matrice **tridiagonale**, la résolution directe peut en réalité se faire en $O(n)$. Pourquoi votre implémentation ne bénéficie-t-elle pas de cette optimisation ?
5. Dans quel cas préféreriez-vous une méthode itérative à une méthode directe ?

**Votre réponse :** *(double-cliquez pour éditer)*


---

# Partie 5 — Perturbations et conditionnement (/3 pts)

En TD2, on a vu qu'une petite perturbation du second membre $b$ pouvait provoquer une **grande** variation de la solution $x$. La sensibilité d'un système linéaire est mesurée par le **nombre de condition** :

$$\kappa(A) = \|A\| \cdot \|A^{-1}\|$$

L'inégalité fondamentale est :

$$\frac{\|\delta x\|}{\|x\|} \leq \kappa(A) \cdot \frac{\|\delta b\|}{\|b\|}$$

Un système avec $\kappa(A) \gg 1$ est **mal conditionné** : les erreurs d'arrondi ou de mesure sont amplifiées.

## Exercice 5.1 — La matrice de Hilbert, un cas extrême (1,5 pt)

La matrice de Hilbert $H_n$ est définie par $H_{ij} = \dfrac{1}{i + j + 1}$ (indices à partir de 0). C'est un exemple classique de matrice **très mal conditionnée**.

**À faire :**
1. Compléter la construction de $H_n$.
2. Pour $n = 5, 8, 10, 12$ : résoudre $H x = b$ avec $b = H \cdot \mathbf{1}$ (la solution exacte est donc $x = \mathbf{1}$).
3. Ajouter une perturbation aléatoire $\delta b$ de norme $10^{-10}$ au second membre et résoudre à nouveau.
4. Mesurer l'erreur relative $\|\delta x\| / \|x\|$ et le nombre de condition $\kappa(H)$.

In [ ]:
def matrice_hilbert(n):
    """Construit la matrice de Hilbert n x n."""
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            H[i, j] = ...  # À COMPLÉTER : 1 / (i + j + 1)
    return H


np.random.seed(42)

print(f"{'n':>4}  {'κ(H)':>14}  {'‖δx‖/‖x‖':>14}  {'κ·‖δb‖/‖b‖':>14}")
print("-" * 52)

for n in [5, 8, 10, 12]:
    H = matrice_hilbert(n)
    x_exact = np.ones(n)
    b = H @ x_exact

    # Perturbation de norme 1e-10
    delta_b = np.random.randn(n)
    delta_b = 1e-10 * delta_b / np.linalg.norm(delta_b)

    x_perturbe = np.linalg.solve(H, b + delta_b)
    delta_x = x_perturbe - x_exact

    kappa = np.linalg.cond(H)
    err_rel_x = np.linalg.norm(delta_x) / np.linalg.norm(x_exact)
    err_rel_b = np.linalg.norm(delta_b) / np.linalg.norm(b)

    print(f"{n:>4}  {kappa:>14.2e}  {err_rel_x:>14.2e}  {kappa * err_rel_b:>14.2e}")

**Votre réponse :** que constatez-vous quand $n$ augmente ? L'inégalité $\|\delta x\|/\|x\| \leq \kappa \cdot \|\delta b\|/\|b\|$ est-elle vérifiée ? Est-il raisonnable de résoudre un système de Hilbert pour $n = 15$ ?

*(double-cliquez pour éditer)*


## Exercice 5.2 — Conditionnement de la matrice de la poutre (1 pt)

On étudie maintenant le conditionnement de la matrice de rigidité $K$ en fonction du nombre de points de discrétisation $n$.

**À faire :**
1. Compléter le code pour calculer $\kappa(K)$ pour différentes valeurs de $n$.
2. Observer comment $\kappa(K)$ évolue avec $n$. Le problème de la poutre est-il bien ou mal conditionné ?

In [ ]:
tailles_cond = [5, 10, 20, 50, 100, 200]
kappas = []

for n in tailles_cond:
    Kn, _ = construire_systeme_poutre(n)
    kappa_n = ...  # À COMPLÉTER : np.linalg.cond(Kn)
    kappas.append(kappa_n)

# --- Tracé fourni ---
plt.figure()
plt.loglog(tailles_cond, kappas, "o-", color="C0", linewidth=2)
plt.xlabel("Taille du système n")
plt.ylabel("Nombre de condition κ(K)")
plt.title("Conditionnement de la matrice de rigidité")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

pente_cond = np.polyfit(np.log(tailles_cond), np.log(kappas), 1)[0]
print(f"Pente en log-log : {pente_cond:.2f}  →  κ(K) croît comme n^{pente_cond:.1f}")

## Exercice 5.3 — Voir l'effet d'une perturbation sur la poutre (0,5 pt)

On perturbe le chargement de la poutre ($n = 50$) par un bruit de $\pm 1\%$ et on observe l'effet sur la solution.

Exécuter le code et répondre : la perturbation de la solution est-elle du même ordre que celle du chargement ? Le nombre de condition de $K$ explique-t-il le résultat ?

In [ ]:
np.random.seed(42)
n_pert = 50
K_pert, f_pert = construire_systeme_poutre(n_pert)
h_pert = L_poutre / (n_pert + 1)

x_ref = np.linalg.solve(K_pert, f_pert)

# Perturbation de ±1% sur le chargement
delta_f = 0.01 * f_pert * np.random.randn(n_pert)
x_pert = np.linalg.solve(K_pert, f_pert + delta_f)

err_rel_f = np.linalg.norm(delta_f) / np.linalg.norm(f_pert)
err_rel_x = np.linalg.norm(x_pert - x_ref) / np.linalg.norm(x_ref)
kappa_K = np.linalg.cond(K_pert)

print(f"Perturbation relative du chargement : {err_rel_f:.4e}")
print(f"Perturbation relative de la solution : {err_rel_x:.4e}")
print(f"Facteur d'amplification observé      : {err_rel_x / err_rel_f:.2f}")
print(f"Nombre de condition κ(K)              : {kappa_K:.2f}")
print(f"Borne théorique κ·‖δf‖/‖f‖           : {kappa_K * err_rel_f:.4e}")

# --- Tracé ---
x_poutre_plot = np.linspace(0, L_poutre, n_pert + 2)
sol_ref = np.concatenate([[0], x_ref, [0]])
sol_pert = np.concatenate([[0], x_pert, [0]])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(x_poutre_plot, sol_ref, "C0", linewidth=2, label="Solution non perturbée")
axes[0].plot(x_poutre_plot, sol_pert, "C1--", linewidth=2, label="Solution perturbée (±1%)")
axes[0].set_xlabel("Position x (m)")
axes[0].set_ylabel("Flèche u (m)")
axes[0].set_title("Effet d'une perturbation du chargement")
axes[0].legend(fontsize=9)
axes[0].grid(True)

axes[1].plot(x_poutre_plot, sol_pert - sol_ref, "C3", linewidth=2)
axes[1].set_xlabel("Position x (m)")
axes[1].set_ylabel("Écart (m)")
axes[1].set_title("Différence perturbée − non perturbée")
axes[1].grid(True)

plt.tight_layout()
plt.show()

**Votre réponse :** *(double-cliquez pour éditer)*


---

## Synthèse — Quand utiliser quelle méthode ?

Le tableau suivant résume ce que vous avez observé. **Complétez-le** à partir de vos résultats.

| Critère | Méthode directe (LU) | Méthode itérative (Jacobi / GS) |
|---------|----------------------|----------------------------------|
| Complexité | $O(n^3)$ pour une matrice dense | Dépend du nombre d'itérations |
| Avantage principal | *(à compléter)* | *(à compléter)* |
| Inconvénient principal | *(à compléter)* | *(à compléter)* |
| Cas d'usage typique | *(à compléter)* | *(à compléter)* |
| Sensibilité au conditionnement | *(à compléter)* | *(à compléter)* |

---

**Fin du TP 2.**

Avant de rendre votre notebook :
1. Vérifiez que toutes les cellules s'exécutent sans erreur (menu *Kernel → Restart & Run All*).
2. Vérifiez que toutes les questions ont une réponse argumentée.
3. Enregistrez le fichier.